In [1]:
from pathlib import Path
import gc
import random
import sys
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
DATA_DIR = Path("/workspace/E-DAIC/original_data")
SPLIT_DIR = Path("/workspace/E-DAIC/splits")
LABEL_DIR = Path("/workspace/E-DAIC/others")
TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VAL_SPLIT_PATH = SPLIT_DIR / "dev_split.csv"
TEST_SPLIT_PATH = SPLIT_DIR / "test_split.csv"
DETAILED_LABEL_PATH = LABEL_DIR / "Detailed_PHQ8_Labels.csv"

TEXT_CACHE_ROOT = Path("/workspace/E-DAIC/cache/phq8_text_embeddings")
AUDIO_CACHE_DIR = Path("/workspace/E-DAIC/cache/phq8_audio_egemaps/egemaps23_turnmean_100hz_turns120_v1")
VIDEO_CACHE_DIR = Path("/workspace/E-DAIC/cache/phq8_video_resnet/resnet2048_turnmean_turns120_v1")

TEXT_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_paperlike")
AUDIO_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_audio_paperlike")
VIDEO_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_video_paperlike")
TAV_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero")

TAV_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

In [3]:
PHQ8_QUESTION_COLUMNS = ["PHQ_8NoInterest", "PHQ_8Depressed", "PHQ_8Sleep", "PHQ_8Tired", "PHQ_8Appetite", "PHQ_8Failure", "PHQ_8Concentrating", "PHQ_8Moving"]
MAX_TURNS = 120
TEXT_FEATURE_DIM = 768
AUDIO_FEATURE_DIM = 23
VIDEO_FEATURE_DIM = 2048
ENCODER_OUTPUT_DIM = 100
FUSED_MODALITY_DIM = ENCODER_OUTPUT_DIM * 2
FINAL_FUSION_DIM = FUSED_MODALITY_DIM * 3
NUM_CLASSES = 4
ATTENTION_HEADS = 4
BATCH_SIZE = 10
NUM_EPOCHS = 20
LEARNING_RATE = 5e-4
ADAM_EPSILON = 1e-8
WEIGHT_DECAY = 1e-3
MAX_GRAD_NORM = 1.0
FUSION_DROPOUT = 0.8
MLP_FIRST_DROPOUT = 0.8
MLP_LAST_DROPOUT = 0.5
ALPHA = 1.0
BETA = 0.5
LOSS_EPSILON = 1e-12
SEEDS = [42, 100, 1234]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
print("Python：", sys.version.split()[0])
print("PyTorch：", torch.__version__)
print("设备：", device)
print("单个编码器输出维度：", ENCODER_OUTPUT_DIM)
print("单个模态融合后维度：", FUSED_MODALITY_DIM)
print("三个模态最终拼接维度：", FINAL_FUSION_DIM)
print("Flatten后维度：", MAX_TURNS * FINAL_FUSION_DIM)
print("Text缓存根目录存在：", TEXT_CACHE_ROOT.exists())
print("Audio缓存目录存在：", AUDIO_CACHE_DIR.exists())
print("Video缓存目录存在：", VIDEO_CACHE_DIR.exists())
print("Text Checkpoint存在：", TEXT_CHECKPOINT_ROOT.exists())
print("Audio Checkpoint存在：", AUDIO_CHECKPOINT_ROOT.exists())
print("Video Checkpoint存在：", VIDEO_CHECKPOINT_ROOT.exists())
print("TAV输出目录：", TAV_CHECKPOINT_ROOT)

Python： 3.12.3
PyTorch： 2.9.0
设备： cuda:0
单个编码器输出维度： 100
单个模态融合后维度： 200
三个模态最终拼接维度： 600
Flatten后维度： 72000
Text缓存根目录存在： True
Audio缓存目录存在： True
Video缓存目录存在： True
Text Checkpoint存在： True
Audio Checkpoint存在： True
Video Checkpoint存在： True
TAV输出目录： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero


In [10]:
text_cache_candidates = sorted(TEXT_CACHE_ROOT.rglob("302.pt"))
print("找到的302文本缓存：", text_cache_candidates)
if len(text_cache_candidates) != 1:
    raise RuntimeError(f"预期找到一个302文本缓存，实际找到{len(text_cache_candidates)}个：{text_cache_candidates}")
TEXT_CACHE_DIR = text_cache_candidates[0].parent
print("正式文本缓存目录：", TEXT_CACHE_DIR)

找到的302文本缓存： [PosixPath('/workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1/302.pt')]
正式文本缓存目录： /workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1


In [7]:
text_cache_path_302 = TEXT_CACHE_DIR / "302.pt"
audio_cache_path_302 = AUDIO_CACHE_DIR / "302.pt"
video_cache_path_302 = VIDEO_CACHE_DIR / "302.pt"
print("302 Text缓存存在：", text_cache_path_302.exists())
print("302 Audio缓存存在：", audio_cache_path_302.exists())
print("302 Video缓存存在：", video_cache_path_302.exists())
text_cache_path_302 = TEXT_CACHE_DIR / "302.pt"
audio_cache_path_302 = AUDIO_CACHE_DIR / "302.pt"
video_cache_path_302 = VIDEO_CACHE_DIR / "302.pt"
print("302 Text缓存存在：", text_cache_path_302.exists())
print("302 Audio缓存存在：", audio_cache_path_302.exists())
print("302 Video缓存存在：", video_cache_path_302.exists())

302 Text缓存存在： True
302 Audio缓存存在： True
302 Video缓存存在： True
302 Text缓存存在： True
302 Audio缓存存在： True
302 Video缓存存在： True


In [15]:
def get_single_modality_best_loss_paths(question_index,seed):
    question_number=question_index+1
    question_name=PHQ8_QUESTION_COLUMNS[question_index]
    text_checkpoint_path=TEXT_CHECKPOINT_ROOT/f"q{question_number}_{question_name}"/f"seed_{seed}"/"best_loss.pt"
    audio_checkpoint_path=AUDIO_CHECKPOINT_ROOT/f"seed_{seed}"/f"question_{question_number}"/"best_loss.pt"
    video_checkpoint_path=VIDEO_CHECKPOINT_ROOT/f"seed_{seed}"/f"question_{question_number}"/"best_loss.pt"
    return text_checkpoint_path,audio_checkpoint_path,video_checkpoint_path

In [19]:
q1_text_best_loss_path, q1_audio_best_loss_path, q1_video_best_loss_path = get_single_modality_best_loss_paths(question_index=0, seed=42)
print("Q1 Text Best Loss：", q1_text_best_loss_path)
print("Q1 Audio Best Loss：", q1_audio_best_loss_path)
print("Q1 Video Best Loss：", q1_video_best_loss_path)
print("Q1 Text Best Loss存在：", q1_text_best_loss_path.exists())
print("Q1 Audio Best Loss存在：", q1_audio_best_loss_path.exists())
print("Q1 Video Best Loss存在：", q1_video_best_loss_path.exists())

Q1 Text Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
Q1 Audio Best Loss： /workspace/E-DAIC/checkpoints/phq8_audio_paperlike/seed_42/question_1/best_loss.pt
Q1 Video Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
Q1 Text Best Loss存在： True
Q1 Audio Best Loss存在： True
Q1 Video Best Loss存在： True


In [20]:
text_cache_302 = torch.load(text_cache_path_302, map_location="cpu", weights_only=False)
audio_cache_302 = torch.load(audio_cache_path_302, map_location="cpu", weights_only=False)
video_cache_302 = torch.load(video_cache_path_302, map_location="cpu", weights_only=False)

In [21]:
text_features_302 = text_cache_302["embeddings"].float()
text_mask_302 = text_cache_302["key_padding_mask"].bool()
audio_features_302 = audio_cache_302["audio_features"].float()
audio_mask_302 = audio_cache_302["padding_mask"].bool()
video_features_302 = video_cache_302["video_features"].float()
video_mask_302 = video_cache_302["padding_mask"].bool()

In [22]:
print("Text特征：", text_features_302.shape)
print("Text Mask：", text_mask_302.shape)
print("Audio特征：", audio_features_302.shape)
print("Audio Mask：", audio_mask_302.shape)
print("Video特征：", video_features_302.shape)
print("Video Mask：", video_mask_302.shape)
print("Text真实轮数：", int((~text_mask_302).sum()))
print("Audio真实轮数：", int((~audio_mask_302).sum()))
print("Video真实轮数：", int((~video_mask_302).sum()))
print("Text全部有限：", bool(torch.isfinite(text_features_302).all()))
print("Audio全部有限：", bool(torch.isfinite(audio_features_302).all()))
print("Video全部有限：", bool(torch.isfinite(video_features_302).all()))

Text特征： torch.Size([120, 768])
Text Mask： torch.Size([120])
Audio特征： torch.Size([120, 23])
Audio Mask： torch.Size([120])
Video特征： torch.Size([120, 2048])
Video Mask： torch.Size([120])
Text真实轮数： 99
Audio真实轮数： 98
Video真实轮数： 98
Text全部有限： True
Audio全部有限： True
Video全部有限： True


In [26]:
text_checkpoint_302 = torch.load(q1_text_best_loss_path, map_location="cpu", weights_only=False)
audio_checkpoint_302 = torch.load(q1_audio_best_loss_path, map_location="cpu", weights_only=False)
video_checkpoint_302 = torch.load(q1_video_best_loss_path, map_location="cpu", weights_only=False)
def validate_single_modality_checkpoint(checkpoint, expected_seed, expected_question_index, expected_modality):
    if checkpoint["seed"] != expected_seed:
        raise RuntimeError(f"{expected_modality} Seed错误：{checkpoint['seed']}")
    if checkpoint["question_index"] != expected_question_index:
        raise RuntimeError(f"{expected_modality}题目索引错误：{checkpoint['question_index']}")
    if checkpoint["checkpoint_type"] != "best_loss":
        raise RuntimeError(f"{expected_modality}类型错误：{checkpoint['checkpoint_type']}")
    print(f"{expected_modality} Checkpoint验证通过")
validate_single_modality_checkpoint(text_checkpoint_302, expected_seed=42, expected_question_index=0, expected_modality="Text")
validate_single_modality_checkpoint(audio_checkpoint_302, expected_seed=42, expected_question_index=0, expected_modality="Audio")
validate_single_modality_checkpoint(video_checkpoint_302, expected_seed=42, expected_question_index=0, expected_modality="Video")

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过


In [27]:
print("Text前10个参数名：", list(text_checkpoint_302["model_state_dict"].keys())[:10])
print("Audio前10个参数名：", list(audio_checkpoint_302["model_state_dict"].keys())[:10])
print("Video前10个参数名：", list(video_checkpoint_302["model_state_dict"].keys())[:10])

Text前10个参数名： ['lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0', 'lstm.weight_ih_l0_reverse', 'lstm.weight_hh_l0_reverse', 'lstm.bias_ih_l0_reverse', 'lstm.bias_hh_l0_reverse', 'attention.in_proj_weight', 'attention.in_proj_bias']
Audio前10个参数名： ['lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0', 'lstm.weight_ih_l0_reverse', 'lstm.weight_hh_l0_reverse', 'lstm.bias_ih_l0_reverse', 'lstm.bias_hh_l0_reverse', 'attention1.in_proj_weight', 'attention1.in_proj_bias']
Video前10个参数名： ['lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0', 'lstm.weight_ih_l0_reverse', 'lstm.weight_hh_l0_reverse', 'lstm.bias_ih_l0_reverse', 'lstm.bias_hh_l0_reverse', 'attention1.in_proj_weight', 'attention1.in_proj_bias']


In [28]:
train_split_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_split_df = pd.read_csv(VAL_SPLIT_PATH)
detailed_label_df = pd.read_csv(DETAILED_LABEL_PATH)
train_metadata_df = train_split_df.merge(detailed_label_df[["Participant_ID"] + PHQ8_QUESTION_COLUMNS], on="Participant_ID", how="left", validate="one_to_one")
val_metadata_df = val_split_df.merge(detailed_label_df[["Participant_ID"] + PHQ8_QUESTION_COLUMNS], on="Participant_ID", how="left", validate="one_to_one")
if train_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise RuntimeError("训练集存在缺失的题目标签")
if val_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise RuntimeError("验证集存在缺失的题目标签")
print("训练参与者数量：", len(train_metadata_df))
print("验证参与者数量：", len(val_metadata_df))
print("训练标签形状：", train_metadata_df[PHQ8_QUESTION_COLUMNS].shape)
print("验证标签形状：", val_metadata_df[PHQ8_QUESTION_COLUMNS].shape)

训练参与者数量： 163
验证参与者数量： 55
训练标签形状： (163, 8)
验证标签形状： (55, 8)


In [30]:
def load_text_audio_video_cache(participant_id):
    participant_id = int(participant_id)
    text_cache_path = TEXT_CACHE_DIR / f"{participant_id}.pt"
    audio_cache_path = AUDIO_CACHE_DIR / f"{participant_id}.pt"
    video_cache_path = VIDEO_CACHE_DIR / f"{participant_id}.pt"
    if not text_cache_path.exists():
        raise FileNotFoundError(f"Text缓存不存在：{text_cache_path}")
    if not audio_cache_path.exists():
        raise FileNotFoundError(f"Audio缓存不存在：{audio_cache_path}")
    if not video_cache_path.exists():
        raise FileNotFoundError(f"Video缓存不存在：{video_cache_path}")
    text_cache = torch.load(text_cache_path, map_location="cpu", weights_only=False)
    audio_cache = torch.load(audio_cache_path, map_location="cpu", weights_only=False)
    video_cache = torch.load(video_cache_path, map_location="cpu", weights_only=False)
    text_features = text_cache["embeddings"].float()
    text_mask = text_cache["key_padding_mask"].bool()
    audio_features = audio_cache["audio_features"].float()
    audio_mask = audio_cache["padding_mask"].bool()
    video_features = video_cache["video_features"].float()
    video_mask = video_cache["padding_mask"].bool()
    if text_features.shape != (MAX_TURNS, TEXT_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id} Text形状错误：{text_features.shape}")
    if audio_features.shape != (MAX_TURNS, AUDIO_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id} Audio形状错误：{audio_features.shape}")
    if video_features.shape != (MAX_TURNS, VIDEO_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id} Video形状错误：{video_features.shape}")
    if text_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id} Text Mask错误：{text_mask.shape}")
    if audio_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id} Audio Mask错误：{audio_mask.shape}")
    if video_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id} Video Mask错误：{video_mask.shape}")
    if not torch.isfinite(text_features).all():
        raise RuntimeError(f"参与者{participant_id} Text包含NaN或Inf")
    if not torch.isfinite(audio_features).all():
        raise RuntimeError(f"参与者{participant_id} Audio包含NaN或Inf")
    if not torch.isfinite(video_features).all():
        raise RuntimeError(f"参与者{participant_id} Video包含NaN或Inf")
    return text_features, text_mask, audio_features, audio_mask, video_features, video_mask

In [29]:
class TextAudioVideoPHQ8Dataset(Dataset):
    def __init__(self, metadata_df, split_name):
        super().__init__()
        self.split_name = split_name
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.labels = torch.tensor(self.metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy(), dtype=torch.long)
        text_feature_list = []
        text_mask_list = []
        audio_feature_list = []
        audio_mask_list = []
        video_feature_list = []
        video_mask_list = []
        for index, participant_id in enumerate(self.participant_ids):
            text_features, text_mask, audio_features, audio_mask, video_features, video_mask = load_text_audio_video_cache(participant_id)
            text_feature_list.append(text_features)
            text_mask_list.append(text_mask)
            audio_feature_list.append(audio_features)
            audio_mask_list.append(audio_mask)
            video_feature_list.append(video_features)
            video_mask_list.append(video_mask)
            if (index + 1) % 50 == 0 or index + 1 == len(self.participant_ids):
                print(f"{split_name}缓存加载进度：{index + 1}/{len(self.participant_ids)}")
        self.text_features = torch.stack(text_feature_list, dim=0)
        self.text_masks = torch.stack(text_mask_list, dim=0)
        self.audio_features = torch.stack(audio_feature_list, dim=0)
        self.audio_masks = torch.stack(audio_mask_list, dim=0)
        self.video_features = torch.stack(video_feature_list, dim=0)
        self.video_masks = torch.stack(video_mask_list, dim=0)

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        return self.text_features[index], self.text_masks[index], self.audio_features[index], self.audio_masks[index], self.video_features[index], self.video_masks[index], self.labels[index]

In [31]:
tav_train_dataset = TextAudioVideoPHQ8Dataset(train_metadata_df, split_name="train")
tav_val_dataset = TextAudioVideoPHQ8Dataset(val_metadata_df, split_name="val")
print("训练Text：", tav_train_dataset.text_features.shape)
print("训练Audio：", tav_train_dataset.audio_features.shape)
print("训练Video：", tav_train_dataset.video_features.shape)
print("训练Text Mask：", tav_train_dataset.text_masks.shape)
print("训练Audio Mask：", tav_train_dataset.audio_masks.shape)
print("训练Video Mask：", tav_train_dataset.video_masks.shape)
print("训练Labels：", tav_train_dataset.labels.shape)

train缓存加载进度：50/163
train缓存加载进度：100/163
train缓存加载进度：150/163
train缓存加载进度：163/163
val缓存加载进度：50/55
val缓存加载进度：55/55
训练Text： torch.Size([163, 120, 768])
训练Audio： torch.Size([163, 120, 23])
训练Video： torch.Size([163, 120, 2048])
训练Text Mask： torch.Size([163, 120])
训练Audio Mask： torch.Size([163, 120])
训练Video Mask： torch.Size([163, 120])
训练Labels： torch.Size([163, 8])


In [32]:
def create_tav_dataloaders(seed):
    train_generator = torch.Generator()
    train_generator.manual_seed(seed)
    train_loader = DataLoader(tav_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False, generator=train_generator)
    val_loader = DataLoader(tav_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    return train_loader, val_loader

In [33]:
tav_train_loader, tav_val_loader = create_tav_dataloaders(seed=42)
batch_text_features, batch_text_masks, batch_audio_features, batch_audio_masks, batch_video_features, batch_video_masks, batch_all_labels = next(iter(tav_train_loader))

In [34]:
print("Text Batch：", batch_text_features.shape)
print("Text Mask：", batch_text_masks.shape)
print("Audio Batch：", batch_audio_features.shape)
print("Audio Mask：", batch_audio_masks.shape)
print("Video Batch：", batch_video_features.shape)
print("Video Mask：", batch_video_masks.shape)
print("Labels：", batch_all_labels.shape)
print("Q1标签：", batch_all_labels[:, 0])

Text Batch： torch.Size([10, 120, 768])
Text Mask： torch.Size([10, 120])
Audio Batch： torch.Size([10, 120, 23])
Audio Mask： torch.Size([10, 120])
Video Batch： torch.Size([10, 120, 2048])
Video Mask： torch.Size([10, 120])
Labels： torch.Size([10, 8])
Q1标签： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])


In [35]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [36]:
class PretrainedTextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=TEXT_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.5, batch_first=True)

    def forward(self, text_features, text_padding_mask):
        lstm_output, _ = self.lstm(text_features)
        text_encoding, _ = self.attention(lstm_output, lstm_output, lstm_output, key_padding_mask=text_padding_mask, need_weights=False)
        return text_encoding

In [37]:
class PretrainedAudioEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=AUDIO_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)

    def forward(self, audio_features, audio_padding_mask):
        lstm_output, _ = self.lstm(audio_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=audio_padding_mask, need_weights=False)
        audio_encoding, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=audio_padding_mask, need_weights=False)
        return audio_encoding

In [38]:
class PretrainedVideoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=VIDEO_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)

    def forward(self, video_features, video_padding_mask):
        lstm_output, _ = self.lstm(video_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=video_padding_mask, need_weights=False)
        video_encoding, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=video_padding_mask, need_weights=False)
        return video_encoding

In [39]:
def extract_state_dict_by_prefixes(full_state_dict, prefixes):
    selected_state_dict = {}
    for parameter_name, parameter_value in full_state_dict.items():
        if any(parameter_name.startswith(prefix) for prefix in prefixes):
            selected_state_dict[parameter_name] = parameter_value
    return selected_state_dict

In [40]:
def load_pretrained_tav_encoders(question_index, seed):
    text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path = get_single_modality_best_loss_paths(question_index, seed)
    text_checkpoint = torch.load(text_checkpoint_path, map_location="cpu", weights_only=False)
    audio_checkpoint = torch.load(audio_checkpoint_path, map_location="cpu", weights_only=False)
    video_checkpoint = torch.load(video_checkpoint_path, map_location="cpu", weights_only=False)
    validate_single_modality_checkpoint(text_checkpoint, seed, question_index, "Text")
    validate_single_modality_checkpoint(audio_checkpoint, seed, question_index, "Audio")
    validate_single_modality_checkpoint(video_checkpoint, seed, question_index, "Video")
    text_encoder = PretrainedTextEncoder()
    audio_encoder = PretrainedAudioEncoder()
    video_encoder = PretrainedVideoEncoder()
    text_encoder_state = extract_state_dict_by_prefixes(text_checkpoint["model_state_dict"], prefixes=("lstm.", "attention."))
    audio_encoder_state = extract_state_dict_by_prefixes(audio_checkpoint["model_state_dict"], prefixes=("lstm.", "attention1.", "attention2."))
    video_encoder_state = extract_state_dict_by_prefixes(video_checkpoint["model_state_dict"], prefixes=("lstm.", "attention1.", "attention2."))
    text_load_result = text_encoder.load_state_dict(text_encoder_state, strict=True)
    audio_load_result = audio_encoder.load_state_dict(audio_encoder_state, strict=True)
    video_load_result = video_encoder.load_state_dict(video_encoder_state, strict=True)
    print("Text保存Epoch：", text_checkpoint["epoch"])
    print("Audio保存Epoch：", audio_checkpoint["epoch"])
    print("Video保存Epoch：", video_checkpoint["epoch"])
    print("Text缺少参数：", text_load_result.missing_keys)
    print("Text多余参数：", text_load_result.unexpected_keys)
    print("Audio缺少参数：", audio_load_result.missing_keys)
    print("Audio多余参数：", audio_load_result.unexpected_keys)
    print("Video缺少参数：", video_load_result.missing_keys)
    print("Video多余参数：", video_load_result.unexpected_keys)
    return text_encoder, audio_encoder, video_encoder, text_checkpoint, audio_checkpoint, video_checkpoint, text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path

In [41]:
set_seed(42)
q1_text_encoder, q1_audio_encoder, q1_video_encoder, q1_text_checkpoint, q1_audio_checkpoint, q1_video_checkpoint, q1_text_checkpoint_path, q1_audio_checkpoint_path, q1_video_checkpoint_path = load_pretrained_tav_encoders(question_index=0, seed=42)

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []


In [42]:
q1_text_encoder = q1_text_encoder.to(device)
q1_audio_encoder = q1_audio_encoder.to(device)
q1_video_encoder = q1_video_encoder.to(device)
q1_text_encoder.eval()
q1_audio_encoder.eval()
q1_video_encoder.eval()

PretrainedVideoEncoder(
  (lstm): LSTM(2048, 50, batch_first=True, bidirectional=True)
  (attention1): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
  )
  (attention2): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
  )
)

In [43]:
with torch.no_grad():
    diagnostic_text_encoding = q1_text_encoder(batch_text_features.to(device), batch_text_masks.to(device))
    diagnostic_audio_encoding = q1_audio_encoder(batch_audio_features.to(device), batch_audio_masks.to(device))
    diagnostic_video_encoding = q1_video_encoder(batch_video_features.to(device), batch_video_masks.to(device))

In [44]:
print("Text编码：", diagnostic_text_encoding.shape)
print("Audio编码：", diagnostic_audio_encoding.shape)
print("Video编码：", diagnostic_video_encoding.shape)
print("Text编码全部有限：", bool(torch.isfinite(diagnostic_text_encoding).all()))
print("Audio编码全部有限：", bool(torch.isfinite(diagnostic_audio_encoding).all()))
print("Video编码全部有限：", bool(torch.isfinite(diagnostic_video_encoding).all()))
print("Text编码器参数：", f"{sum(parameter.numel() for parameter in q1_text_encoder.parameters()):,}")
print("Audio编码器参数：", f"{sum(parameter.numel() for parameter in q1_audio_encoder.parameters()):,}")
print("Video编码器参数：", f"{sum(parameter.numel() for parameter in q1_video_encoder.parameters()):,}")

Text编码： torch.Size([10, 120, 100])
Audio编码： torch.Size([10, 120, 100])
Video编码： torch.Size([10, 120, 100])
Text编码全部有限： True
Audio编码全部有限： True
Video编码全部有限： True
Text编码器参数： 368,400
Audio编码器参数： 110,800
Video编码器参数： 920,800


In [53]:
class TextAudioVideoQuestMF(nn.Module):
    def __init__(self, text_encoder, audio_encoder, video_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.audio_encoder = audio_encoder
        self.video_encoder = video_encoder
        self.audio_to_text_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.video_to_text_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_to_audio_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.video_to_audio_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_to_video_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.audio_to_video_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_self_attention = nn.MultiheadAttention(embed_dim=FUSED_MODALITY_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.audio_self_attention = nn.MultiheadAttention(embed_dim=FUSED_MODALITY_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.video_self_attention = nn.MultiheadAttention(embed_dim=FUSED_MODALITY_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(MLP_FIRST_DROPOUT), nn.Linear(MAX_TURNS * FINAL_FUSION_DIM, 256), nn.ReLU(), nn.Dropout(MLP_LAST_DROPOUT), nn.Linear(256, NUM_CLASSES))
        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False
        self.text_encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self
    def forward(self, text_features, text_padding_mask, audio_features, audio_padding_mask, video_features, video_padding_mask, return_intermediates=False):
        text_encoding = self.text_encoder(text_features, text_padding_mask)
        audio_encoding = self.audio_encoder(audio_features, audio_padding_mask)
        video_encoding = self.video_encoder(video_features, video_padding_mask)
        audio_to_text, _ = self.audio_to_text_cross_attention(text_encoding, audio_encoding, audio_encoding, key_padding_mask=audio_padding_mask, need_weights=False)
        video_to_text, _ = self.video_to_text_cross_attention(text_encoding, video_encoding, video_encoding, key_padding_mask=video_padding_mask, need_weights=False)
        text_cross_fused = torch.cat((audio_to_text, video_to_text), dim=2)
        audio_to_text, _ = self.audio_to_text_cross_attention(text_encoding, audio_encoding, audio_encoding, key_padding_mask=audio_padding_mask, need_weights=False)
        video_to_text, _ = self.video_to_text_cross_attention(text_encoding, video_encoding, video_encoding, key_padding_mask=video_padding_mask, need_weights=False)
        text_cross_fused = torch.cat((audio_to_text, video_to_text), dim=2)
        text_to_audio, _ = self.text_to_audio_cross_attention(audio_encoding, text_encoding, text_encoding, key_padding_mask=text_padding_mask, need_weights=False)
        video_to_audio, _ = self.video_to_audio_cross_attention(audio_encoding, video_encoding, video_encoding, key_padding_mask=video_padding_mask, need_weights=False)
        audio_cross_fused = torch.cat((text_to_audio, video_to_audio), dim=2)
        text_to_video, _ = self.text_to_video_cross_attention(video_encoding, text_encoding, text_encoding, key_padding_mask=text_padding_mask, need_weights=False)
        audio_to_video, _ = self.audio_to_video_cross_attention(video_encoding, audio_encoding, audio_encoding, key_padding_mask=audio_padding_mask, need_weights=False)
        video_cross_fused = torch.cat((text_to_video, audio_to_video), dim=2)
        text_fused_encoding, _ = self.text_self_attention(text_cross_fused, text_cross_fused, text_cross_fused, key_padding_mask=text_padding_mask, need_weights=False)
        audio_fused_encoding, _ = self.audio_self_attention(audio_cross_fused, audio_cross_fused, audio_cross_fused, key_padding_mask=audio_padding_mask, need_weights=False)
        video_fused_encoding, _ = self.video_self_attention(video_cross_fused, video_cross_fused, video_cross_fused, key_padding_mask=video_padding_mask, need_weights=False)
        final_fused_encoding = torch.cat((text_fused_encoding, audio_fused_encoding, video_fused_encoding), dim=2)
        logits = self.mlp(final_fused_encoding)
        if return_intermediates:
            intermediates = {}
            intermediates["text_encoding"] = text_encoding
            intermediates["audio_encoding"] = audio_encoding
            intermediates["video_encoding"] = video_encoding
            intermediates["audio_to_text"] = audio_to_text
            intermediates["video_to_text"] = video_to_text
            intermediates["text_to_audio"] = text_to_audio
            intermediates["video_to_audio"] = video_to_audio
            intermediates["text_to_video"] = text_to_video
            intermediates["audio_to_video"] = audio_to_video
            intermediates["text_cross_fused"] = text_cross_fused
            intermediates["audio_cross_fused"] = audio_cross_fused
            intermediates["video_cross_fused"] = video_cross_fused
            intermediates["text_fused_encoding"] = text_fused_encoding
            intermediates["audio_fused_encoding"] = audio_fused_encoding
            intermediates["video_fused_encoding"] = video_fused_encoding
            intermediates["final_fused_encoding"] = final_fused_encoding
            return logits, intermediates
        return logits


In [54]:
set_seed(42)
q1_text_encoder, q1_audio_encoder, q1_video_encoder, q1_text_checkpoint, q1_audio_checkpoint, q1_video_checkpoint, q1_text_checkpoint_path, q1_audio_checkpoint_path, q1_video_checkpoint_path = load_pretrained_tav_encoders(question_index=0, seed=42)
q1_tav_model = TextAudioVideoQuestMF(q1_text_encoder, q1_audio_encoder, q1_video_encoder).to(device)
q1_tav_model.train()
text_trainable_parameters = sum(parameter.numel() for parameter in q1_tav_model.text_encoder.parameters() if parameter.requires_grad)
audio_trainable_parameters = sum(parameter.numel() for parameter in q1_tav_model.audio_encoder.parameters() if parameter.requires_grad)
video_trainable_parameters = sum(parameter.numel() for parameter in q1_tav_model.video_encoder.parameters() if parameter.requires_grad)
total_trainable_parameters = sum(parameter.numel() for parameter in q1_tav_model.parameters() if parameter.requires_grad)
print("Text可训练参数：", f"{text_trainable_parameters:,}")
print("Audio可训练参数：", f"{audio_trainable_parameters:,}")
print("Video可训练参数：", f"{video_trainable_parameters:,}")
print("模型总可训练参数：", f"{total_trainable_parameters:,}")
print("Text训练模式：", q1_tav_model.text_encoder.training)
print("Audio训练模式：", q1_tav_model.audio_encoder.training)
print("Video训练模式：", q1_tav_model.video_encoder.training)

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Text可训练参数： 0
Audio可训练参数： 110,800
Video可训练参数： 920,800
模型总可训练参数： 20,189,684
Text训练模式： False
Audio训练模式： True
Video训练模式： True


In [55]:
q1_tav_model.eval()
with torch.no_grad():
    q1_tav_logits, q1_tav_intermediates = q1_tav_model(batch_text_features.to(device), batch_text_masks.to(device), batch_audio_features.to(device), batch_audio_masks.to(device), batch_video_features.to(device), batch_video_masks.to(device), return_intermediates=True)
print("Text编码：", q1_tav_intermediates["text_encoding"].shape)
print("Audio编码：", q1_tav_intermediates["audio_encoding"].shape)
print("Video编码：", q1_tav_intermediates["video_encoding"].shape)
print("Text Cross融合：", q1_tav_intermediates["text_cross_fused"].shape)
print("Audio Cross融合：", q1_tav_intermediates["audio_cross_fused"].shape)
print("Video Cross融合：", q1_tav_intermediates["video_cross_fused"].shape)
print("最终融合：", q1_tav_intermediates["final_fused_encoding"].shape)
print("最终Logits：", q1_tav_logits.shape)
print("Logits全部有限：", bool(torch.isfinite(q1_tav_logits).all()))

Text编码： torch.Size([10, 120, 100])
Audio编码： torch.Size([10, 120, 100])
Video编码： torch.Size([10, 120, 100])
Text Cross融合： torch.Size([10, 120, 200])
Audio Cross融合： torch.Size([10, 120, 200])
Video Cross融合： torch.Size([10, 120, 200])
最终融合： torch.Size([10, 120, 600])
最终Logits： torch.Size([10, 4])
Logits全部有限： True


In [56]:
class ImbOLLLoss(nn.Module):
    def __init__(self, class_weights, alpha=1.0, epsilon=1e-12):
        super().__init__()
        self.register_buffer("class_weights", class_weights.float())
        self.alpha = alpha
        self.epsilon = epsilon

    def forward(self, logits, targets):
        probabilities = torch.softmax(logits, dim=1)
        class_indices = torch.arange(logits.shape[1], device=logits.device).view(1, -1)
        ordinal_distances = torch.abs(targets.view(-1, 1) - class_indices).float()
        sample_weights = self.class_weights[targets].view(-1, 1)
        weighted_distances = torch.pow(sample_weights * ordinal_distances, self.alpha)
        losses = -torch.log(1.0 - probabilities + self.epsilon) * weighted_distances
        return losses.sum(dim=1).mean()

In [57]:
def calculate_gradient_statistics(parameters):
    squared_gradient_sum = 0.0
    gradient_tensor_count = 0
    for parameter in parameters:
        if parameter.grad is not None:
            squared_gradient_sum += parameter.grad.detach().pow(2).sum().item()
            gradient_tensor_count += 1
    return squared_gradient_sum ** 0.5, gradient_tensor_count

In [58]:
q1_train_labels = tav_train_dataset.labels[:, 0]
q1_class_counts = torch.bincount(q1_train_labels, minlength=NUM_CLASSES)
q1_class_weights = torch.pow(len(q1_train_labels) / q1_class_counts.float(), BETA)
q1_criterion = ImbOLLLoss(q1_class_weights, alpha=ALPHA).to(device)
q1_optimizer = torch.optim.AdamW((parameter for parameter in q1_tav_model.parameters() if parameter.requires_grad), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
q1_tav_model.train()
q1_optimizer.zero_grad(set_to_none=True)
diagnostic_logits = q1_tav_model(batch_text_features.to(device), batch_text_masks.to(device), batch_audio_features.to(device), batch_audio_masks.to(device), batch_video_features.to(device), batch_video_masks.to(device))
diagnostic_targets = batch_all_labels[:, 0].to(device)
diagnostic_loss = q1_criterion(diagnostic_logits, diagnostic_targets)
diagnostic_loss.backward()

In [59]:
text_gradient_norm, text_gradient_count = calculate_gradient_statistics(q1_tav_model.text_encoder.parameters())
audio_gradient_norm, audio_gradient_count = calculate_gradient_statistics(q1_tav_model.audio_encoder.parameters())
video_gradient_norm, video_gradient_count = calculate_gradient_statistics(q1_tav_model.video_encoder.parameters())
fusion_parameters = [parameter for name, parameter in q1_tav_model.named_parameters() if not name.startswith(("text_encoder.", "audio_encoder.", "video_encoder."))]
fusion_gradient_norm, fusion_gradient_count = calculate_gradient_statistics(fusion_parameters)
print("ImbOLL：", diagnostic_loss.item())
print("Text梯度：", text_gradient_norm, "张量数：", text_gradient_count)
print("Audio梯度：", audio_gradient_norm, "张量数：", audio_gradient_count)
print("Video梯度：", video_gradient_norm, "张量数：", video_gradient_count)
print("融合层梯度：", fusion_gradient_norm, "张量数：", fusion_gradient_count)

ImbOLL： 2.2597131729125977
Text梯度： 0.0 张量数： 0
Audio梯度： 0.10996090520211597 张量数： 16
Video梯度： 0.033924439019880244 张量数： 16
融合层梯度： 1.2980884886728683 张量数： 40


In [60]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

In [61]:
def calculate_ccc(predictions, targets, epsilon=1e-12):
    predictions = predictions.double()
    targets = targets.double()
    prediction_mean = predictions.mean()
    target_mean = targets.mean()
    covariance = ((predictions - prediction_mean) * (targets - target_mean)).mean()
    prediction_variance = ((predictions - prediction_mean) ** 2).mean()
    target_variance = ((targets - target_mean) ** 2).mean()
    ccc = (2.0 * covariance) / (prediction_variance + target_variance + (prediction_mean - target_mean) ** 2 + epsilon)
    return float(ccc.item())

In [62]:
def calculate_question_metrics(predictions, targets):
    predictions = predictions.long()
    targets = targets.long()
    predictions_numpy = predictions.numpy()
    targets_numpy = targets.numpy()
    accuracy = accuracy_score(targets_numpy, predictions_numpy)
    micro_f1 = f1_score(targets_numpy, predictions_numpy, labels=list(range(NUM_CLASSES)), average="micro", zero_division=0)
    macro_f1 = f1_score(targets_numpy, predictions_numpy, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)
    weighted_f1 = f1_score(targets_numpy, predictions_numpy, labels=list(range(NUM_CLASSES)), average="weighted", zero_division=0)
    matrix = confusion_matrix(targets_numpy, predictions_numpy, labels=list(range(NUM_CLASSES)))
    rmse = float(torch.sqrt(torch.mean((predictions.float() - targets.float()) ** 2)).item())
    mae = float(torch.mean(torch.abs(predictions.float() - targets.float())).item())
    ccc = calculate_ccc(predictions, targets)
    return {"accuracy": accuracy, "micro_f1": micro_f1, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "ccc": ccc, "rmse": rmse, "mae": mae, "confusion_matrix": torch.tensor(matrix)}

In [63]:
def run_tav_epoch(model, data_loader, question_index, criterion, optimizer=None):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_samples = 0
    prediction_list = []
    target_list = []
    for text_features, text_masks, audio_features, audio_masks, video_features, video_masks, all_labels in data_loader:
        text_features = text_features.to(device)
        text_masks = text_masks.to(device)
        audio_features = audio_features.to(device)
        audio_masks = audio_masks.to(device)
        video_features = video_features.to(device)
        video_masks = video_masks.to(device)
        targets = all_labels[:, question_index].to(device)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(text_features, text_masks, audio_features, audio_masks, video_features, video_masks)
            loss = criterion(logits, targets)
            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
        batch_size = targets.shape[0]
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        prediction_list.append(logits.argmax(dim=1).detach().cpu())
        target_list.append(targets.detach().cpu())
    predictions = torch.cat(prediction_list)
    targets = torch.cat(target_list)
    metrics = calculate_question_metrics(predictions, targets)
    metrics["loss"] = total_loss / total_samples
    metrics["predictions"] = predictions
    metrics["targets"] = targets
    return metrics

In [64]:
del q1_tav_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [65]:
set_seed(42)
tav_train_loader, tav_val_loader = create_tav_dataloaders(seed=42)
q1_text_encoder, q1_audio_encoder, q1_video_encoder, q1_text_checkpoint, q1_audio_checkpoint, q1_video_checkpoint, q1_text_checkpoint_path, q1_audio_checkpoint_path, q1_video_checkpoint_path = load_pretrained_tav_encoders(question_index=0, seed=42)
q1_tav_model = TextAudioVideoQuestMF(q1_text_encoder, q1_audio_encoder, q1_video_encoder).to(device)
q1_train_labels = tav_train_dataset.labels[:, 0]
q1_class_counts = torch.bincount(q1_train_labels, minlength=NUM_CLASSES)
q1_class_weights = torch.pow(len(q1_train_labels) / q1_class_counts.float(), BETA)
q1_criterion = ImbOLLLoss(q1_class_weights, alpha=ALPHA).to(device)
q1_optimizer = torch.optim.AdamW((parameter for parameter in q1_tav_model.parameters() if parameter.requires_grad), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []


In [66]:
q1_train_metrics = run_tav_epoch(q1_tav_model, tav_train_loader, question_index=0, criterion=q1_criterion, optimizer=q1_optimizer)
q1_val_metrics = run_tav_epoch(q1_tav_model, tav_val_loader, question_index=0, criterion=q1_criterion)

In [67]:
print(f"Train Loss：{q1_train_metrics['loss']:.6f}")
print(f"Train CCC：{q1_train_metrics['ccc']:.6f}")
print(f"Train Accuracy：{q1_train_metrics['accuracy']:.6f}")
print(f"Val Loss：{q1_val_metrics['loss']:.6f}")
print(f"Val CCC：{q1_val_metrics['ccc']:.6f}")
print(f"Val Accuracy：{q1_val_metrics['accuracy']:.6f}")
print(f"Val RMSE：{q1_val_metrics['rmse']:.6f}")
print(f"Val MAE：{q1_val_metrics['mae']:.6f}")
print("Val混淆矩阵：")
print(q1_val_metrics["confusion_matrix"])

Train Loss：2.006856
Train CCC：0.408225
Train Accuracy：0.503067
Val Loss：2.558830
Val CCC：0.302995
Val Accuracy：0.418182
Val RMSE：1.095445
Val MAE：0.763636
Val混淆矩阵：
tensor([[19,  1,  5,  0],
        [ 8,  1, 13,  0],
        [ 1,  0,  3,  0],
        [ 2,  0,  2,  0]])


In [68]:
def compact_metrics(metrics):
    return {"loss": float(metrics["loss"]), "accuracy": float(metrics["accuracy"]), "micro_f1": float(metrics["micro_f1"]), "macro_f1": float(metrics["macro_f1"]), "weighted_f1": float(metrics["weighted_f1"]), "ccc": float(metrics["ccc"]), "rmse": float(metrics["rmse"]), "mae": float(metrics["mae"]), "confusion_matrix": metrics["confusion_matrix"].tolist()}

In [69]:
def save_tav_checkpoint(path, checkpoint_type, epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path):
    checkpoint = {}
    checkpoint["checkpoint_type"] = checkpoint_type
    checkpoint["epoch"] = epoch
    checkpoint["question_index"] = question_index
    checkpoint["question_number"] = question_index + 1
    checkpoint["question_name"] = PHQ8_QUESTION_COLUMNS[question_index]
    checkpoint["seed"] = seed
    checkpoint["model_state_dict"] = model.state_dict()
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
    checkpoint["train_metrics"] = compact_metrics(train_metrics)
    checkpoint["val_metrics"] = compact_metrics(val_metrics)
    checkpoint["class_counts"] = class_counts.tolist()
    checkpoint["class_weights"] = class_weights.tolist()
    checkpoint["best_val_loss"] = float(best_val_loss)
    checkpoint["best_val_ccc"] = float(best_val_ccc)
    checkpoint["best_loss_epoch"] = int(best_loss_epoch)
    checkpoint["best_ccc_epoch"] = int(best_ccc_epoch)
    checkpoint["text_pretrained_checkpoint"] = str(text_checkpoint_path)
    checkpoint["audio_pretrained_checkpoint"] = str(audio_checkpoint_path)
    checkpoint["video_pretrained_checkpoint"] = str(video_checkpoint_path)
    checkpoint["fusion_variant"] = "three_modality_nopostzero"
    checkpoint["config"] = {"max_turns": MAX_TURNS, "text_feature_dim": TEXT_FEATURE_DIM, "audio_feature_dim": AUDIO_FEATURE_DIM, "video_feature_dim": VIDEO_FEATURE_DIM, "encoder_output_dim": ENCODER_OUTPUT_DIM, "per_target_fusion_dim": FUSED_MODALITY_DIM, "final_fusion_dim": FINAL_FUSION_DIM, "attention_heads": ATTENTION_HEADS, "fusion_dropout": FUSION_DROPOUT, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "alpha": ALPHA, "beta": BETA, "text_frozen": True, "postzero": False}
    torch.save(checkpoint, path)

In [70]:
def train_tav_question(seed, question_index):
    set_seed(seed)
    train_loader, val_loader = create_tav_dataloaders(seed)
    text_encoder, audio_encoder, video_encoder, text_checkpoint, audio_checkpoint, video_checkpoint, text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path = load_pretrained_tav_encoders(question_index, seed)
    model = TextAudioVideoQuestMF(text_encoder, audio_encoder, video_encoder).to(device)
    train_labels = tav_train_dataset.labels[:, question_index]
    class_counts = torch.bincount(train_labels, minlength=NUM_CLASSES)
    class_weights = torch.pow(len(train_labels) / class_counts.float(), BETA)
    criterion = ImbOLLLoss(class_weights, alpha=ALPHA).to(device)
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    checkpoint_dir = TAV_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = checkpoint_dir / "best_loss.pt"
    best_ccc_path = checkpoint_dir / "best_ccc.pt"
    last_path = checkpoint_dir / "last.pt"
    history_path = checkpoint_dir / "history.csv"
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = -1
    best_ccc_epoch = -1
    history_rows = []
    print("=" * 100)
    print(f"开始训练T+A+V：Seed={seed}，Q{question_index + 1}，{PHQ8_QUESTION_COLUMNS[question_index]}")
    print("类别数量：", class_counts.tolist())
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.perf_counter()
        train_metrics = run_tav_epoch(model, train_loader, question_index, criterion, optimizer)
        val_metrics = run_tav_epoch(model, val_loader, question_index, criterion)
        elapsed_seconds = time.perf_counter() - epoch_start_time
        improved_loss = val_metrics["loss"] < best_val_loss
        improved_ccc = val_metrics["ccc"] > best_val_ccc
        if improved_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
        if improved_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
        if improved_loss:
            save_tav_checkpoint(best_loss_path, "best_loss", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path)
        if improved_ccc:
            save_tav_checkpoint(best_ccc_path, "best_ccc", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path)
        history_rows.append({"Epoch": epoch, "Train_Loss": train_metrics["loss"], "Train_CCC": train_metrics["ccc"], "Train_Accuracy": train_metrics["accuracy"], "Val_Loss": val_metrics["loss"], "Val_CCC": val_metrics["ccc"], "Val_Accuracy": val_metrics["accuracy"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Elapsed_Seconds": elapsed_seconds})
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_metrics['loss']:.6f} | Val Loss {val_metrics['loss']:.6f} | Val CCC {val_metrics['ccc']:.6f} | Val Acc {val_metrics['accuracy']:.6f} | {elapsed_seconds:.2f}s")
    save_tav_checkpoint(last_path, "last", NUM_EPOCHS, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, audio_checkpoint_path, video_checkpoint_path)
    pd.DataFrame(history_rows).to_csv(history_path, index=False)
    print("=" * 100)
    print(f"训练完成：Seed={seed}，Q{question_index + 1}")
    print(f"最佳Val Loss：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    return {"seed": seed, "question_index": question_index, "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}

In [71]:
tav_q1_seed42_result = train_tav_question(seed=42, question_index=0)

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
开始训练T+A+V：Seed=42，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
Epoch 01/20 | Train Loss 2.006856 | Val Loss 2.558830 | Val CCC 0.302995 | Val Acc 0.418182 | 1.75s
Epoch 02/20 | Train Loss 1.353113 | Val Loss 3.167231 | Val CCC 0.301075 | Val Acc 0.509091 | 1.50s
Epoch 03/20 | Train Loss 1.315467 | Val Loss 2.404885 | Val CCC 0.402174 | Val Acc 0.527273 | 1.48s
Epoch 04/20 | Train Loss 1.151039 | Val Loss 2.720844 | Val CCC 0.366395 | Val Acc 0.618182 | 1.39s
Epoch 05/20 | Train Loss 1.049124 | Val Loss 3.566317 | Val CCC 0.283608 | Val Acc 0.490909 | 1.38s
Epoch 06/20 | Train Loss 1.170496 | Val Loss 3.037740 | Val CCC 0.326187 | Val Acc 0.527273 | 1.37s
Epoch 07/20 | Train Loss 1.224841 | Val Loss 3.473085 | Val CCC 0.327217 | Val Acc 0.527273 | 1.50s
Epoch 08/20 | Train Loss 1.253058 | Val Loss 2.5

In [72]:
q1_tav_best_ccc_path = TAV_CHECKPOINT_ROOT / "seed_42" / "question_1" / "best_ccc.pt"
q1_tav_saved_checkpoint = torch.load(q1_tav_best_ccc_path, map_location="cpu", weights_only=False)
q1_tav_loaded_items = load_pretrained_tav_encoders(question_index=0, seed=42)
q1_tav_restored_text_encoder = q1_tav_loaded_items[0]
q1_tav_restored_audio_encoder = q1_tav_loaded_items[1]
q1_tav_restored_video_encoder = q1_tav_loaded_items[2]
q1_tav_restored_model = TextAudioVideoQuestMF(q1_tav_restored_text_encoder, q1_tav_restored_audio_encoder, q1_tav_restored_video_encoder).to(device)
q1_tav_load_result = q1_tav_restored_model.load_state_dict(q1_tav_saved_checkpoint["model_state_dict"], strict=True)
q1_tav_saved_class_weights = torch.as_tensor(q1_tav_saved_checkpoint["class_weights"], dtype=torch.float32, device=device)
q1_tav_restored_criterion = ImbOLLLoss(q1_tav_saved_class_weights)
_, q1_tav_restore_val_loader = create_tav_dataloaders(seed=42)
q1_tav_restored_metrics = run_tav_epoch(q1_tav_restored_model, q1_tav_restore_val_loader, 0, q1_tav_restored_criterion, optimizer=None)
print("Checkpoint：", q1_tav_best_ccc_path)
print("Checkpoint类型：", q1_tav_saved_checkpoint["checkpoint_type"])
print("保存Epoch：", q1_tav_saved_checkpoint["epoch"])
print("缺少参数：", q1_tav_load_result.missing_keys)
print("多余参数：", q1_tav_load_result.unexpected_keys)
for metric_name in ["loss", "accuracy", "micro_f1", "macro_f1", "weighted_f1", "ccc", "rmse", "mae"]:
    saved_value = float(q1_tav_saved_checkpoint["val_metrics"][metric_name])
    restored_value = float(q1_tav_restored_metrics[metric_name])
    difference = abs(saved_value - restored_value)
    print(f"{metric_name:>12} | 保存 {saved_value:.9f} | 恢复 {restored_value:.9f} | 差值 {difference:.3e}")
    assert difference < 1e-6
print("T+A+V Q1 Checkpoint恢复验证通过")

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/seed_42/question_1/best_ccc.pt
Checkpoint类型： best_ccc
保存Epoch： 17
缺少参数： []
多余参数： []
        loss | 保存 2.911822818 | 恢复 2.911822818 | 差值 0.000e+00
    accuracy | 保存 0.600000000 | 恢复 0.600000000 | 差值 0.000e+00
    micro_f1 | 保存 0.600000000 | 恢复 0.600000000 | 差值 0.000e+00
    macro_f1 | 保存 0.383134413 | 恢复 0.383134413 | 差值 0.000e+00
 weighted_f1 | 保存 0.580929500 | 恢复 0.580929500 | 差值 0.000e+00
         ccc | 保存 0.485415520 | 恢复 0.485415520 | 差值 0.000e+00
        rmse | 保存 0.786245406 | 恢复 0.786245406 | 差值 0.000e+00
         mae | 保存 0.472727269 | 恢复 0.472727269 | 差值 0.000e+00
T+A+V Q1 Checkpoint恢复验证通过


In [74]:
tav_seed42_results = []

for question_index in range(1, 8):
    current_result = train_tav_question(seed=42, question_index=question_index)
    tav_seed42_results.append(current_result)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 9
Audio保存Epoch： 11
Video保存Epoch： 8
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
开始训练T+A+V：Seed=42，Q2，PHQ_8Depressed
类别数量： [68, 63, 18, 14]
Epoch 01/20 | Train Loss 2.206376 | Val Loss 2.053315 | Val CCC 0.360000 | Val Acc 0.418182 | 1.60s
Epoch 02/20 | Train Loss 1.661960 | Val Loss 2.038682 | Val CCC 0.581463 | Val Acc 0.509091 | 1.56s
Epoch 03/20 | Train Loss 1.808136 | Val Loss 2.046854 | Val CCC 0.373510 | Val Acc 0.527273 | 1.40s
Epoch 04/20 | Train Loss 1.585796 | Val Loss 2.019823 | Val CCC 0.551268 | Val Acc 0.490909 | 1.90s
Epoch 05/20 | Train Loss 1.603923 | Val Loss 2.027655 | Val CCC 0.536703 | Val Acc 0.527273 | 1.39s
Epoch 06/20 | Train Loss 1.690335 | Val Loss 1.969212 | Val CCC 0.568627 | Val Acc 0.490909 | 1.37s
Epoch 07/20 | Train Loss 1.568156 | Val Loss 1.980830 | Val CCC 0.498331 | Val Acc 0.527273 | 1.37s
Epoch 08/20 | Train Loss 1.561943 | Val Loss 2.09

In [75]:
tav_seed42_question_rows = []

for question_index, question_name in enumerate(PHQ8_QUESTION_COLUMNS):
    checkpoint_path = TAV_CHECKPOINT_ROOT / "seed_42" / f"question_{question_index + 1}" / "best_ccc.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    val_metrics = checkpoint["val_metrics"]
    row = {}
    row["Question_Number"] = question_index + 1
    row["Question_Name"] = question_name
    row["Checkpoint_Epoch"] = checkpoint["epoch"]
    row["CCC"] = val_metrics["ccc"]
    row["RMSE"] = val_metrics["rmse"]
    row["MAE"] = val_metrics["mae"]
    row["Accuracy"] = val_metrics["accuracy"]
    tav_seed42_question_rows.append(row)

tav_seed42_question_summary_df = pd.DataFrame(tav_seed42_question_rows)
display(tav_seed42_question_summary_df)

,Question_Number,Question_Name,Checkpoint_Epoch,CCC,RMSE,MAE,Accuracy
0,1,PHQ_8NoInterest,17,0.485416,0.786245,0.472727,0.600000
1,2,PHQ_8Depressed,2,0.581463,0.842075,0.563636,0.509091
2,3,PHQ_8Sleep,13,0.487288,1.095445,0.763636,0.436364
3,4,PHQ_8Tired,19,0.426975,1.000000,0.745455,0.381818
4,5,PHQ_8Appetite,10,0.551549,0.904534,0.600000,0.509091
5,6,PHQ_8Failure,2,0.559312,0.863397,0.600000,0.472727
6,7,PHQ_8Concentrating,3,0.592928,0.809040,0.545455,0.509091
7,8,PHQ_8Moving,10,0.332828,0.762770,0.472727,0.581818


In [76]:
for question_index in range(8):
    question_dir = TAV_CHECKPOINT_ROOT / "seed_42" / f"question_{question_index + 1}"
    best_loss_exists = (question_dir / "best_loss.pt").exists()
    best_ccc_exists = (question_dir / "best_ccc.pt").exists()
    last_exists = (question_dir / "last.pt").exists()
    history_exists = (question_dir / "history.csv").exists()
    print(f"Q{question_index + 1} | Best Loss {best_loss_exists} | Best CCC {best_ccc_exists} | Last {last_exists} | History {history_exists}")

Q1 | Best Loss True | Best CCC True | Last True | History True
Q2 | Best Loss True | Best CCC True | Last True | History True
Q3 | Best Loss True | Best CCC True | Last True | History True
Q4 | Best Loss True | Best CCC True | Last True | History True
Q5 | Best Loss True | Best CCC True | Last True | History True
Q6 | Best Loss True | Best CCC True | Last True | History True
Q7 | Best Loss True | Best CCC True | Last True | History True
Q8 | Best Loss True | Best CCC True | Last True | History True


In [81]:
def evaluate_tav_seed_on_validation(seed):
    _, validation_loader = create_tav_dataloaders(seed=seed)
    question_prediction_list = []
    question_true_list = []

    for question_index in range(8):
        checkpoint_path = TAV_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        loaded_items = load_pretrained_tav_encoders(question_index=question_index, seed=seed)
        text_encoder = loaded_items[0]
        audio_encoder = loaded_items[1]
        video_encoder = loaded_items[2]
        model = TextAudioVideoQuestMF(text_encoder, audio_encoder, video_encoder).to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        model.eval()
        prediction_chunks = []
        true_chunks = []

        with torch.no_grad():
            for batch in validation_loader:
                text_features, text_masks, audio_features, audio_masks, video_features, video_masks, all_labels = batch
                logits = model(text_features.to(device), text_masks.to(device), audio_features.to(device), audio_masks.to(device), video_features.to(device), video_masks.to(device))
                predictions = torch.argmax(logits, dim=1).cpu()
                true_labels = all_labels[:, question_index].cpu()
                prediction_chunks.append(predictions)
                true_chunks.append(true_labels)

        question_predictions = torch.cat(prediction_chunks, dim=0)
        question_true = torch.cat(true_chunks, dim=0)
        question_prediction_list.append(question_predictions)
        question_true_list.append(question_true)
        print(f"Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | 样本 {len(question_predictions)} | 参数完整 {len(load_result.missing_keys) == 0 and len(load_result.unexpected_keys) == 0}")
        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    predicted_question_matrix = torch.stack(question_prediction_list, dim=1).numpy()
    true_question_matrix = torch.stack(question_true_list, dim=1).numpy()
    predicted_totals = predicted_question_matrix.sum(axis=1)
    true_totals = true_question_matrix.sum(axis=1)
    total_ccc = float(calculate_ccc(torch.from_numpy(predicted_totals), torch.from_numpy(true_totals)))
    total_rmse = float(np.sqrt(np.mean((predicted_totals - true_totals) ** 2)))
    total_mae = float(np.mean(np.abs(predicted_totals - true_totals)))
    exact_accuracy = float(np.mean(predicted_totals == true_totals))
    participant_ids = val_metadata_df["Participant_ID"].astype(int).to_numpy()
    prediction_df = pd.DataFrame({"Participant_ID": participant_ids})

    for question_index in range(8):
        prediction_df[f"Predicted_Q{question_index + 1}"] = predicted_question_matrix[:, question_index]
        prediction_df[f"True_Q{question_index + 1}"] = true_question_matrix[:, question_index]

    prediction_df["Predicted_Total"] = predicted_totals
    prediction_df["True_Total"] = true_totals
    prediction_path = TAV_CHECKPOINT_ROOT / f"seed_{seed}_validation_predictions.csv"
    prediction_df.to_csv(prediction_path, index=False)
    result = {}
    result["seed"] = seed
    result["sample_count"] = len(prediction_df)
    result["ccc"] = total_ccc
    result["rmse"] = total_rmse
    result["mae"] = total_mae
    result["exact_accuracy"] = exact_accuracy
    result["prediction_path"] = prediction_path
    result["prediction_df"] = prediction_df
    return result

In [82]:
tav_seed42_validation_result = evaluate_tav_seed_on_validation(seed=42)
print("=" * 70)
print("验证参与者数量：", tav_seed42_validation_result["sample_count"])
print(f"总分CCC：{tav_seed42_validation_result['ccc']:.6f}")
print(f"总分RMSE：{tav_seed42_validation_result['rmse']:.6f}")
print(f"总分MAE：{tav_seed42_validation_result['mae']:.6f}")
print(f"完全相等比例：{tav_seed42_validation_result['exact_accuracy']:.6f}")
print("预测总分范围：", tav_seed42_validation_result["prediction_df"]["Predicted_Total"].min(), "～", tav_seed42_validation_result["prediction_df"]["Predicted_Total"].max())
print("真实总分范围：", tav_seed42_validation_result["prediction_df"]["True_Total"].min(), "～", tav_seed42_validation_result["prediction_df"]["True_Total"].max())
print("预测文件：", tav_seed42_validation_result["prediction_path"])

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q1 | Epoch 17 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 9
Audio保存Epoch： 11
Video保存Epoch： 8
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q2 | Epoch 02 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 9
Audio保存Epoch： 14
Video保存Epoch： 14
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q3 | Epoch 13 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 11
Audio保存Epoch： 28
Video保存Epoch： 7
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q4 | Epoch 19 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 3

In [83]:
print("验证参与者数量：", tav_seed42_validation_result["sample_count"])
print(f"总分CCC：{tav_seed42_validation_result['ccc']:.6f}")
print(f"总分RMSE：{tav_seed42_validation_result['rmse']:.6f}")
print(f"总分MAE：{tav_seed42_validation_result['mae']:.6f}")
print(f"完全相等比例：{tav_seed42_validation_result['exact_accuracy']:.6f}")

验证参与者数量： 55
总分CCC：0.710736
总分RMSE：4.297991
总分MAE：3.309091
完全相等比例：0.072727


In [84]:
tav_remaining_seed_results = {}

for seed in [100, 1234]:
    seed_results = []
    print("=" * 100)
    print(f"开始完整训练 Seed={seed}")
    print("=" * 100)

    for question_index in range(8):
        current_result = train_tav_question(seed=seed, question_index=question_index)
        seed_results.append(current_result)
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tav_remaining_seed_results[seed] = seed_results

开始完整训练 Seed=100
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 10
Audio保存Epoch： 8
Video保存Epoch： 14
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
开始训练T+A+V：Seed=100，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
Epoch 01/20 | Train Loss 1.850259 | Val Loss 3.728074 | Val CCC 0.462073 | Val Acc 0.490909 | 1.61s
Epoch 02/20 | Train Loss 1.673218 | Val Loss 4.422473 | Val CCC 0.119191 | Val Acc 0.545455 | 1.68s
Epoch 03/20 | Train Loss 1.514468 | Val Loss 3.713111 | Val CCC 0.279369 | Val Acc 0.490909 | 2.09s
Epoch 04/20 | Train Loss 1.757857 | Val Loss 4.124222 | Val CCC 0.265955 | Val Acc 0.509091 | 1.62s
Epoch 05/20 | Train Loss 1.671832 | Val Loss 2.786579 | Val CCC 0.286848 | Val Acc 0.472727 | 1.66s
Epoch 06/20 | Train Loss 1.407892 | Val Loss 2.999182 | Val CCC 0.413368 | Val Acc 0.545455 | 1.48s
Epoch 07/20 | Train Loss 1.438666 | Val Loss 2.843956 | Val CCC 0.365496 | Val Acc 0.563636 | 1.58s
Epoch 08/20 | Train Loss 1.4134

In [85]:
for seed in SEEDS:
    best_loss_flags = []
    best_ccc_flags = []
    last_flags = []
    history_flags = []

    for question_index in range(8):
        question_dir = TAV_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}"
        best_loss_flags.append((question_dir / "best_loss.pt").exists())
        best_ccc_flags.append((question_dir / "best_ccc.pt").exists())
        last_flags.append((question_dir / "last.pt").exists())
        history_flags.append((question_dir / "history.csv").exists())

    print("=" * 70)
    print("Seed：", seed)
    print("问题数量：", len(best_ccc_flags))
    print("Best Loss全部存在：", all(best_loss_flags))
    print("Best CCC全部存在：", all(best_ccc_flags))
    print("Last全部存在：", all(last_flags))
    print("History全部存在：", all(history_flags))

Seed： 42
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 100
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 1234
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [86]:
tav_seed100_validation_result = evaluate_tav_seed_on_validation(seed=100)
tav_seed1234_validation_result = evaluate_tav_seed_on_validation(seed=1234)
tav_validation_results = [tav_seed42_validation_result, tav_seed100_validation_result, tav_seed1234_validation_result]
tav_validation_rows = []

for result in tav_validation_results:
    row = {}
    row["Seed"] = result["seed"]
    row["Sample_Count"] = result["sample_count"]
    row["CCC"] = result["ccc"]
    row["RMSE"] = result["rmse"]
    row["MAE"] = result["mae"]
    row["Exact_Accuracy"] = result["exact_accuracy"]
    tav_validation_rows.append(row)

tav_validation_seed_df = pd.DataFrame(tav_validation_rows)
display(tav_validation_seed_df)

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 10
Audio保存Epoch： 8
Video保存Epoch： 14
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q1 | Epoch 08 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 8
Audio保存Epoch： 43
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q2 | Epoch 19 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 15
Video保存Epoch： 9
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q3 | Epoch 03 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 8
Audio保存Epoch： 47
Video保存Epoch： 6
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Q4 | Epoch 10 | 样本 55 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 11
Audio保存Epoch： 7


,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,55,0.710736,4.297991,3.309091,0.072727
1,100,55,0.655306,4.494441,3.472727,0.072727
2,1234,55,0.676513,4.498485,3.654545,0.072727


In [87]:
tav_validation_summary_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    row = {}
    row["Metric"] = metric_name
    row["Mean"] = tav_validation_seed_df[metric_name].mean()
    row["Std"] = tav_validation_seed_df[metric_name].std(ddof=1)
    tav_validation_summary_rows.append(row)

tav_validation_summary_df = pd.DataFrame(tav_validation_summary_rows)
display(tav_validation_summary_df)

,Metric,Mean,Std
0,CCC,0.680852,0.027969
1,RMSE,4.430306,0.114606
2,MAE,3.478788,0.172807
3,Exact_Accuracy,0.072727,0.000000


In [88]:
tav_validation_seed_path = TAV_CHECKPOINT_ROOT / "three_seed_validation_results.csv"
tav_validation_summary_path = TAV_CHECKPOINT_ROOT / "three_seed_validation_summary.csv"
tav_validation_seed_df.to_csv(tav_validation_seed_path, index=False)
tav_validation_summary_df.to_csv(tav_validation_summary_path, index=False)
print("逐Seed验证结果：", tav_validation_seed_path)
print("三Seed验证汇总：", tav_validation_summary_path)

逐Seed验证结果： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/three_seed_validation_results.csv
三Seed验证汇总： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/three_seed_validation_summary.csv


In [89]:
class TextAudioVideoPHQ8TestDataset(Dataset):
    def __init__(self, metadata_df):
        super().__init__()
        self.metadata_df = metadata_df.reset_index(drop=True).copy()

        if "PHQ8_Score" in self.metadata_df.columns:
            score_column = "PHQ8_Score"
        elif "PHQ_Score" in self.metadata_df.columns:
            score_column = "PHQ_Score"
        else:
            raise KeyError(f"测试数据缺少PHQ总分字段，现有字段：{self.metadata_df.columns.tolist()}")

        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.true_totals = torch.tensor(self.metadata_df[score_column].to_numpy(), dtype=torch.long)
        text_feature_list = []
        text_mask_list = []
        audio_feature_list = []
        audio_mask_list = []
        video_feature_list = []
        video_mask_list = []

        for index, participant_id in enumerate(self.participant_ids):
            text_features, text_mask, audio_features, audio_mask, video_features, video_mask = load_text_audio_video_cache(participant_id)
            text_feature_list.append(text_features)
            text_mask_list.append(text_mask)
            audio_feature_list.append(audio_features)
            audio_mask_list.append(audio_mask)
            video_feature_list.append(video_features)
            video_mask_list.append(video_mask)

            if (index + 1) % 20 == 0 or index + 1 == len(self.participant_ids):
                print(f"test Dataset加载进度：{index + 1}/{len(self.participant_ids)}")

        self.text_features = torch.stack(text_feature_list, dim=0)
        self.text_masks = torch.stack(text_mask_list, dim=0)
        self.audio_features = torch.stack(audio_feature_list, dim=0)
        self.audio_masks = torch.stack(audio_mask_list, dim=0)
        self.video_features = torch.stack(video_feature_list, dim=0)
        self.video_masks = torch.stack(video_mask_list, dim=0)

        if self.text_features.shape != (len(self), MAX_TURNS, TEXT_FEATURE_DIM):
            raise RuntimeError(f"测试文本特征形状错误：{self.text_features.shape}")

        if self.audio_features.shape != (len(self), MAX_TURNS, AUDIO_FEATURE_DIM):
            raise RuntimeError(f"测试语音特征形状错误：{self.audio_features.shape}")

        if self.video_features.shape != (len(self), MAX_TURNS, VIDEO_FEATURE_DIM):
            raise RuntimeError(f"测试视频特征形状错误：{self.video_features.shape}")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        participant_id = torch.tensor(self.participant_ids[index], dtype=torch.long)
        return self.text_features[index], self.text_masks[index], self.audio_features[index], self.audio_masks[index], self.video_features[index], self.video_masks[index], self.true_totals[index], participant_id

In [91]:
test_metadata_df = pd.read_csv(TEST_SPLIT_PATH)
test_metadata_df["Participant_ID"] = test_metadata_df["Participant_ID"].astype(int)
print("测试集路径：", TEST_SPLIT_PATH)
print("测试参与者数量：", len(test_metadata_df))
print("测试集字段：", test_metadata_df.columns.tolist())
print("参与者是否唯一：", test_metadata_df["Participant_ID"].is_unique)
print("总分是否缺失：", test_metadata_df["PHQ_Score"].isna().any())

测试集路径： /workspace/E-DAIC/splits/test_split.csv
测试参与者数量： 56
测试集字段： ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']
参与者是否唯一： True
总分是否缺失： False


In [92]:
tav_test_dataset = TextAudioVideoPHQ8TestDataset(test_metadata_df)
tav_test_loader = DataLoader(tav_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
test_batch = next(iter(tav_test_loader))
test_text_features, test_text_masks, test_audio_features, test_audio_masks, test_video_features, test_video_masks, test_true_totals, test_participant_ids = test_batch
print("测试参与者数量：", len(tav_test_dataset))
print("测试批次数：", len(tav_test_loader))
print("测试文本特征：", test_text_features.shape)
print("测试文本Mask：", test_text_masks.shape)
print("测试语音特征：", test_audio_features.shape)
print("测试语音Mask：", test_audio_masks.shape)
print("测试视频特征：", test_video_features.shape)
print("测试视频Mask：", test_video_masks.shape)
print("测试真实总分：", test_true_totals.shape)
print("测试参与者：", test_participant_ids)
print("文本全部有限：", torch.isfinite(test_text_features).all().item())
print("语音全部有限：", torch.isfinite(test_audio_features).all().item())
print("视频全部有限：", torch.isfinite(test_video_features).all().item())
print("真实总分范围：", test_true_totals.min().item(), "～", test_true_totals.max().item())

test Dataset加载进度：20/56
test Dataset加载进度：40/56
test Dataset加载进度：56/56
测试参与者数量： 56
测试批次数： 6
测试文本特征： torch.Size([10, 120, 768])
测试文本Mask： torch.Size([10, 120])
测试语音特征： torch.Size([10, 120, 23])
测试语音Mask： torch.Size([10, 120])
测试视频特征： torch.Size([10, 120, 2048])
测试视频Mask： torch.Size([10, 120])
测试真实总分： torch.Size([10])
测试参与者： tensor([600, 602, 604, 605, 606, 607, 609, 615, 618, 619])
文本全部有限： True
语音全部有限： True
视频全部有限： True
真实总分范围： 0 ～ 13


In [93]:
print("完整测试集真实总分范围：", tav_test_dataset.true_totals.min().item(), "～", tav_test_dataset.true_totals.max().item())

完整测试集真实总分范围： 0 ～ 22


In [94]:
def evaluate_tav_seed_on_test(seed):
    question_prediction_list = []
    reference_true_totals = None
    reference_participant_ids = None

    for question_index in range(8):
        checkpoint_path = TAV_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        loaded_items = load_pretrained_tav_encoders(question_index=question_index, seed=seed)
        text_encoder = loaded_items[0]
        audio_encoder = loaded_items[1]
        video_encoder = loaded_items[2]
        model = TextAudioVideoQuestMF(text_encoder, audio_encoder, video_encoder).to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        model.eval()
        prediction_chunks = []
        true_total_chunks = []
        participant_id_chunks = []

        with torch.no_grad():
            for batch in tav_test_loader:
                text_features, text_masks, audio_features, audio_masks, video_features, video_masks, true_totals, participant_ids = batch
                logits = model(text_features.to(device), text_masks.to(device), audio_features.to(device), audio_masks.to(device), video_features.to(device), video_masks.to(device))
                predictions = torch.argmax(logits, dim=1).cpu()
                prediction_chunks.append(predictions)
                true_total_chunks.append(true_totals.cpu())
                participant_id_chunks.append(participant_ids.cpu())

        question_predictions = torch.cat(prediction_chunks, dim=0)
        current_true_totals = torch.cat(true_total_chunks, dim=0)
        current_participant_ids = torch.cat(participant_id_chunks, dim=0)
        question_prediction_list.append(question_predictions)

        if reference_true_totals is None:
            reference_true_totals = current_true_totals
            reference_participant_ids = current_participant_ids
        else:
            assert torch.equal(reference_true_totals, current_true_totals)
            assert torch.equal(reference_participant_ids, current_participant_ids)

        parameters_complete = len(load_result.missing_keys) == 0 and len(load_result.unexpected_keys) == 0
        print(f"Seed {seed} | Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | 样本 {len(question_predictions)} | 参数完整 {parameters_complete}")
        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    predicted_question_matrix = torch.stack(question_prediction_list, dim=1)
    predicted_totals = predicted_question_matrix.sum(dim=1)
    total_ccc = float(calculate_ccc(predicted_totals, reference_true_totals))
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.double() - reference_true_totals.double()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.double() - reference_true_totals.double())).item())
    exact_accuracy = float(torch.mean((predicted_totals == reference_true_totals).double()).item())
    prediction_df = pd.DataFrame({"Participant_ID": reference_participant_ids.numpy()})

    for question_index in range(8):
        prediction_df[f"Predicted_Q{question_index + 1}"] = predicted_question_matrix[:, question_index].numpy()

    prediction_df["Predicted_Total"] = predicted_totals.numpy()
    prediction_df["True_Total"] = reference_true_totals.numpy()
    prediction_path = TAV_CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_56.csv"
    prediction_df.to_csv(prediction_path, index=False)
    result = {}
    result["seed"] = seed
    result["sample_count"] = len(prediction_df)
    result["ccc"] = total_ccc
    result["rmse"] = total_rmse
    result["mae"] = total_mae
    result["exact_accuracy"] = exact_accuracy
    result["predicted_min"] = int(predicted_totals.min().item())
    result["predicted_max"] = int(predicted_totals.max().item())
    result["true_min"] = int(reference_true_totals.min().item())
    result["true_max"] = int(reference_true_totals.max().item())
    result["prediction_path"] = prediction_path
    return result

In [95]:
tav_test_results = []

for seed in SEEDS:
    seed_result = evaluate_tav_seed_on_test(seed=seed)
    tav_test_results.append(seed_result)
    print("=" * 80)
    print("Seed：", seed)
    print("测试人数：", seed_result["sample_count"])
    print(f"CCC：{seed_result['ccc']:.6f}")
    print(f"RMSE：{seed_result['rmse']:.6f}")
    print(f"MAE：{seed_result['mae']:.6f}")
    print(f"完全相等比例：{seed_result['exact_accuracy']:.6f}")
    print("预测范围：", seed_result["predicted_min"], "～", seed_result["predicted_max"])
    print("真实范围：", seed_result["true_min"], "～", seed_result["true_max"])

Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 13
Audio保存Epoch： 10
Video保存Epoch： 2
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Seed 42 | Q1 | Epoch 17 | 样本 56 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 9
Audio保存Epoch： 11
Video保存Epoch： 8
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Seed 42 | Q2 | Epoch 02 | 样本 56 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 9
Audio保存Epoch： 14
Video保存Epoch： 14
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Seed 42 | Q3 | Epoch 13 | 样本 56 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkpoint验证通过
Text保存Epoch： 11
Audio保存Epoch： 28
Video保存Epoch： 7
Text缺少参数： []
Text多余参数： []
Audio缺少参数： []
Audio多余参数： []
Video缺少参数： []
Video多余参数： []
Seed 42 | Q4 | Epoch 19 | 样本 56 | 参数完整 True
Text Checkpoint验证通过
Audio Checkpoint验证通过
Video Checkp

In [96]:
tav_test_rows = []

for result in tav_test_results:
    row = {}
    row["Seed"] = result["seed"]
    row["Sample_Count"] = result["sample_count"]
    row["CCC"] = result["ccc"]
    row["RMSE"] = result["rmse"]
    row["MAE"] = result["mae"]
    row["Exact_Accuracy"] = result["exact_accuracy"]
    tav_test_rows.append(row)

tav_test_seed_df = pd.DataFrame(tav_test_rows)
display(tav_test_seed_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,56,0.620632,6.037088,4.589286,0.142857
1,100,56,0.637954,5.335193,4.178571,0.089286
2,1234,56,0.684662,4.895333,3.750000,0.071429


In [97]:
tav_test_summary_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    row = {}
    row["Metric"] = metric_name
    row["Mean"] = tav_test_seed_df[metric_name].mean()
    row["Std"] = tav_test_seed_df[metric_name].std(ddof=1)
    tav_test_summary_rows.append(row)

tav_test_summary_df = pd.DataFrame(tav_test_summary_rows)
display(tav_test_summary_df)

,Metric,Mean,Std
0,CCC,0.647749,0.033120
1,RMSE,5.422538,0.575867
2,MAE,4.172619,0.419675
3,Exact_Accuracy,0.101190,0.037173


In [98]:
tav_test_seed_path = TAV_CHECKPOINT_ROOT / "three_seed_test_results_56.csv"
tav_test_summary_path = TAV_CHECKPOINT_ROOT / "three_seed_test_summary_56.csv"
tav_test_seed_df.to_csv(tav_test_seed_path, index=False)
tav_test_summary_df.to_csv(tav_test_summary_path, index=False)
print("逐Seed测试结果：", tav_test_seed_path)
print("三Seed测试汇总：", tav_test_summary_path)

逐Seed测试结果： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/three_seed_test_results_56.csv
三Seed测试汇总： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/three_seed_test_summary_56.csv


In [99]:
VIDEO_EXCLUDED_TEST_ID = 637
tav_candidate55_rows = []

for seed in SEEDS:
    prediction_path = TAV_CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_56.csv"
    prediction_df = pd.read_csv(prediction_path)
    candidate55_df = prediction_df[prediction_df["Participant_ID"] != VIDEO_EXCLUDED_TEST_ID].reset_index(drop=True)
    predicted_totals = torch.tensor(candidate55_df["Predicted_Total"].to_numpy(), dtype=torch.float64)
    true_totals = torch.tensor(candidate55_df["True_Total"].to_numpy(), dtype=torch.float64)
    ccc = float(calculate_ccc(predicted_totals, true_totals))
    rmse = float(torch.sqrt(torch.mean((predicted_totals - true_totals) ** 2)).item())
    mae = float(torch.mean(torch.abs(predicted_totals - true_totals)).item())
    exact_accuracy = float(torch.mean((predicted_totals == true_totals).double()).item())
    row = {}
    row["Seed"] = seed
    row["Sample_Count"] = len(candidate55_df)
    row["Excluded_ID"] = VIDEO_EXCLUDED_TEST_ID
    row["CCC"] = ccc
    row["RMSE"] = rmse
    row["MAE"] = mae
    row["Exact_Accuracy"] = exact_accuracy
    tav_candidate55_rows.append(row)

tav_candidate55_seed_df = pd.DataFrame(tav_candidate55_rows)
display(tav_candidate55_seed_df)

,Seed,Sample_Count,Excluded_ID,CCC,RMSE,MAE,Exact_Accuracy
0,42,55,637,0.609042,6.067799,4.6,0.145455
1,100,55,637,0.623071,5.368257,4.2,0.090909
2,1234,55,637,0.672867,4.937795,3.8,0.072727


In [100]:
tav_candidate55_summary_rows = []

for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    row = {}
    row["Metric"] = metric_name
    row["Mean"] = tav_candidate55_seed_df[metric_name].mean()
    row["Std"] = tav_candidate55_seed_df[metric_name].std(ddof=1)
    tav_candidate55_summary_rows.append(row)

tav_candidate55_summary_df = pd.DataFrame(tav_candidate55_summary_rows)
display(tav_candidate55_summary_df)

,Metric,Mean,Std
0,CCC,0.634993,0.033542
1,RMSE,5.457950,0.570316
2,MAE,4.200000,0.400000
3,Exact_Accuracy,0.103030,0.037848


In [101]:
tav_candidate55_seed_path = TAV_CHECKPOINT_ROOT / "three_seed_test_results_candidate55.csv"
tav_candidate55_summary_path = TAV_CHECKPOINT_ROOT / "three_seed_test_summary_candidate55.csv"
tav_candidate55_seed_df.to_csv(tav_candidate55_seed_path, index=False)
tav_candidate55_summary_df.to_csv(tav_candidate55_summary_path, index=False)
print("Candidate55逐Seed结果：", tav_candidate55_seed_path)
print("Candidate55汇总结果：", tav_candidate55_summary_path)

Candidate55逐Seed结果： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/three_seed_test_results_candidate55.csv
Candidate55汇总结果： /workspace/E-DAIC/checkpoints/phq8_text_audio_video_nopostzero/three_seed_test_summary_candidate55.csv
